# Neighbourhood Vibe Mapper — Sentiment Fine-Tuning
LoRA fine-tune DistilBERT on Yelp reviews for 3-class sentiment.  
Run on GPU (T4 or P100). Output: `sentiment_model.zip` in `/kaggle/working/`.

In [ ]:
!pip install -q peft transformers accelerate orjson

In [ ]:
import os, shutil, zipfile
from pathlib import Path

import numpy as np
import torch
import orjson
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME        = "distilbert-base-uncased"
MAX_TRAIN_SAMPLES = 200_000   # ~20 min on T4; increase to 0 for all 7M (hours)
TRAIN_EPOCHS      = 3
TRAIN_BATCH_SIZE  = 32
LEARNING_RATE     = 2e-4
MAX_SEQ_LENGTH    = 128       # 128 is plenty for Yelp reviews; 256 wastes compute

YELP_REVIEW_PATH  = Path("/kaggle/input/yelp-dataset/yelp_academic_dataset_review.json")
OUTPUT_DIR        = Path("/kaggle/working/artifacts/sentiment_model")
CHECKPOINTS_DIR   = Path("/kaggle/working/artifacts/sentiment_checkpoints")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: no GPU found — this will take many hours on CPU")

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
def stars_to_label(stars: int) -> int:
    if stars <= 2: return 0   # negative
    if stars == 3: return 1   # neutral
    return 2                  # positive

texts, labels = [], []
with open(YELP_REVIEW_PATH, "rb") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = orjson.loads(line)
        except Exception:
            continue
        text  = rec.get("text")
        stars = rec.get("stars")
        if not text or stars is None:
            continue
        texts.append(text)
        labels.append(stars_to_label(int(stars)))
        if MAX_TRAIN_SAMPLES and len(texts) >= MAX_TRAIN_SAMPLES:
            break

print(f"Loaded {len(texts):,} reviews")
print(f"Label distribution: {np.bincount(labels).tolist()}")

In [ ]:
# ── Tokenize ──────────────────────────────────────────────────────────────────
from transformers import DataCollatorWithPadding
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ds = Dataset.from_dict({"text": texts, "label": labels})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

ds = ds.map(tokenize, batched=True, remove_columns=["text"])
ds = ds.rename_column("label", "labels")
ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer)  # dynamic padding = faster
print(f"Dataset ready: {len(ds):,} examples")

In [ ]:
# ── Model + LoRA ──────────────────────────────────────────────────────────────
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Weighted loss trainer ─────────────────────────────────────────────────────
label_counts = np.bincount(labels, minlength=3).astype(np.float64)
label_counts = np.maximum(label_counts, 1.0)
class_weights = torch.tensor(
    (1.0 / label_counts) / (1.0 / label_counts).sum() * 3.0,
    dtype=torch.float32,
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_val = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )(outputs.logits, labels_val)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
total_steps  = (len(ds) // TRAIN_BATCH_SIZE) * TRAIN_EPOCHS
warmup_steps = int(total_steps * 0.1)
print(f"total_steps={total_steps:,}  warmup_steps={warmup_steps:,}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    logging_steps=100,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    fp16=(device == "cuda"),   # half-precision on GPU for extra speed
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
)
trainer.train()

In [ ]:
# ── Merge LoRA + save ─────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
merged = model.merge_and_unload()
merged.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Model saved to {OUTPUT_DIR}")

# Zip for easy download
zip_path = "/kaggle/working/sentiment_model.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.rglob("*"):
        zf.write(f, f.relative_to(OUTPUT_DIR.parent))

print(f"Zipped to {zip_path}")
print("Download via: File → Download output → sentiment_model.zip")

# Cleanup checkpoints
if CHECKPOINTS_DIR.exists():
    shutil.rmtree(CHECKPOINTS_DIR)